# Lab 16: Fast Fourier Transforms

Independent-study notebook with solutions.

This notebook accompanies Chapter 16. It studies the discrete Fourier transform (DFT), Fourier matrix, FFT recursion, and basic frequency filtering.

## Learning goals

1. Construct the Fourier matrix $F_n$.
2. Verify $F_n^*F_n=nI_n$.
3. Compare matrix DFT with `np.fft.fft`.
4. Implement a simple recursive FFT.
5. Use FFT coefficients for filtering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## 1. Fourier matrix

For $\omega_n=e^{-2\pi i/n}$, define

$$
F_n=[\omega_n^{jk}]_{j,k=0}^{n-1}.
$$

In [ ]:
def fourier_matrix(n):
    omega = np.exp(-2j*np.pi/n)
    return np.array([[omega**(j*k) for k in range(n)] for j in range(n)], dtype=complex)

F4 = fourier_matrix(4)
F4

### Question 1

Verify by computation that $F_4^*F_4=4I$.

In [ ]:
# Solution
F4.conj().T @ F4

In [ ]:
np.allclose(F4.conj().T @ F4, 4*np.eye(4))

## 2. DFT of basic signals

Compute the DFT of impulse, constant, and alternating signals.

In [ ]:
signals = {
    "impulse": np.array([1,0,0,0], dtype=complex),
    "constant": np.array([1,1,1,1], dtype=complex),
    "alternating": np.array([1,-1,1,-1], dtype=complex),
}

for name, x in signals.items():
    print(name, "->", F4 @ x)

### Interpretation

- The impulse has all frequencies equally.
- The constant signal has only zero frequency.
- The alternating signal has frequency index $k=2$ for $n=4$.

## 3. Matrix DFT versus NumPy FFT

In [ ]:
f = np.array([1, 2, 0, -1, 0, 1, 0, 0], dtype=complex)
F8 = fourier_matrix(8)
fhat_matrix = F8 @ f
fhat_numpy = np.fft.fft(f)

print(fhat_matrix)
print(fhat_numpy)
print("Same?", np.allclose(fhat_matrix, fhat_numpy))

### Question 2

Recover $f$ from its DFT using the formula

$$
f=\frac1nF_n^*\widehat f.
$$

In [ ]:
# Solution
n = len(f)
f_recovered = (1/n) * F8.conj().T @ fhat_matrix
print(f_recovered)
print("Recovered?", np.allclose(f_recovered, f))

## 4. Recursive FFT

The FFT splits the input into even and odd samples.

In [ ]:
def fft_recursive(x):
    x = np.asarray(x, dtype=complex)
    n = len(x)
    if n == 1:
        return x
    if n % 2 != 0:
        raise ValueError("This simple implementation requires length a power of 2.")
    even = fft_recursive(x[::2])
    odd = fft_recursive(x[1::2])
    twiddle = np.exp(-2j*np.pi*np.arange(n//2)/n)
    top = even + twiddle * odd
    bottom = even - twiddle * odd
    return np.concatenate([top, bottom])

print(fft_recursive(f))
print("Same as NumPy?", np.allclose(fft_recursive(f), np.fft.fft(f)))

### Question 3

Explain why the recursion gives $O(n\log n)$ complexity.

**Solution.** At each level, the algorithm solves two DFT problems of size $n/2$ and uses $O(n)$ work to combine them. Therefore

$$
T(n)=2T(n/2)+O(n).
$$

There are $\log_2 n$ levels and each level costs $O(n)$ total work, so $T(n)=O(n\log n)$.

## 5. Visualizing frequency magnitudes

In [ ]:
n = 64
t = np.linspace(0, 1, n, endpoint=False)
signal = np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*12*t)
F = np.fft.fft(signal)

plt.figure(figsize=(8, 4))
plt.plot(t, signal)
plt.title("Time-domain signal")
plt.xlabel("time")
plt.ylabel("amplitude")
plt.show()

plt.figure(figsize=(8, 4))
plt.stem(np.arange(n), np.abs(F))
plt.title("Frequency magnitudes")
plt.xlabel("frequency index")
plt.ylabel("magnitude")
plt.show()

### Question 4

Which frequency indices are largest?

**Solution.** You should see peaks at $k=5$ and $k=12$, and also at their conjugate-symmetric partners $n-5$ and $n-12$ because the signal is real-valued.

## 6. Low-pass filtering

In [ ]:
filtered_F = F.copy()
# Keep only low frequencies up to 7 and their symmetric partners.
filtered_F[8:n-7] = 0
filtered_signal = np.fft.ifft(filtered_F).real

plt.figure(figsize=(8, 4))
plt.plot(t, signal, label="original")
plt.plot(t, filtered_signal, label="filtered")
plt.legend()
plt.title("Low-pass filtering")
plt.xlabel("time")
plt.ylabel("amplitude")
plt.show()

### Question 5

Which component remains after this filter?

**Solution.** The frequency $5$ component remains, while the frequency $12$ component is removed.

## 7. Challenge: circulant matrices

A circulant matrix represents circular convolution. Fourier vectors are eigenvectors of circular shifts, so the Fourier basis diagonalizes every circulant matrix.

In [ ]:
def circulant_from_first_col(c):
    c = np.asarray(c)
    n = len(c)
    C = np.zeros((n,n), dtype=complex)
    for j in range(n):
        C[:,j] = np.roll(c, j)
    return C

c = np.array([2, -1, 0, -1], dtype=complex)
C = circulant_from_first_col(c)
F4 = fourier_matrix(4)
Lambda = F4 @ C @ ((1/4)*F4.conj().T)
print("C=")
print(C)
print("Diagonalized form F C F^{-1}=")
print(np.round(Lambda, 8))

The result is diagonal up to rounding error, illustrating Fourier diagonalization of circulant matrices.